# <b>Task 2.1 - Executive Summary.</b>

## Objective.
Discover interpretable market classes that separate Binance BTC perpetual futures maker trades by expected markout (30s, 120s, 300s) and turnover. Data covers ~6 months (Nov 2025 – Apr 2026), split into train (Nov–Jan) and validation (Feb–Apr).

## Unconditional Baseline.
Weighted maker PnL across all trades: **-0.105 bps (30s), -0.037 bps (120s), +0.009 bps (300s)**. The maker suffers adverse selection on short horizons and only breaks even at 300s including the 0.5 bps rebate. The goal is to find classes that isolate trades with markout clearly different from this baseline.

## Key Findings.

### Strongest classes (markout spread between best and worst bucket).

**Distance to mid - strongest single feature (~0.56 bps at 30s).** Bucket 1 (trades far below mid, aggressive taker sells) earns +0.29 bps (30s) rising to +0.57 bps (300s). All other buckets are negative or near zero. Interpretation: when a taker sells aggressively below mid, the maker buys at a discount and profits from mean reversion. The relationship is non-monotonic (U-shaped), so a tree-based model is required.

**VPIN - cleanest monotonic signal (~0.34 bps at 30s).** Bucket 1 (low toxicity) earns +0.09 → +0.19 bps across horizons, bucket 5 (high toxicity) loses -0.245 -> -0.144 bps. This directly confirms Easley et al. (2012): higher probability of informed trading systematically harms the maker. Turnover is evenly distributed (~18–22% per bucket), making it highly usable.

**Order book imbalance (~0.28 bps at 300s).** Bucket 5 (bid-heavy) earns +0.20 bps at 300s, bucket 1 (ask-heavy) loses -0.176 bps at 30s. Monotonic across horizons.

**Liquidation flow.** KS-tests show bucket 5 (high liquidation volume) has the strongest distributional shift (KS up to 0.084 at 20s, p ≈ 0). Liquidation activity from both exchanges (Bybit shifted +200ms) carries unique information not duplicated by price-based features.

### Non-linear (V-shaped) classes.
**Spread and volatility** show that moderate levels (bucket 4) are most toxic - the transition zone where adverse selection peaks — while extreme levels (bucket 5) flip to positive markout (post-shock reversion). Volatility KS-statistics are the highest of all features (0.21 at bucket 1, 60s window), confirming a strong distributional effect.

### Intraday seasonality.
**Hour of day** shows large variation: hour 22 UTC is worst (-0.73 bps at 30s), late-European/US hours (19 UTC) are best (+0.23 bps at 300s). Adverse selection is time-dependent and should be a model feature.

### Redundancy found.
**Microprice deviation is mathematically equivalent to OBI on this dataset.** With only best bid/ask available, `(microprice - mid) / half_spread = 2 × OBI - 1`, producing identical buckets and identical markout. Microprice would only add value with deeper order book levels (L2/L3). This is a data limitation.

### Weakest class.
**OFI** showed the noisiest, least monotonic pattern of the literature features, likely because order-flow imbalance computed from LOB-only updates is too noisy without depth.

## Stability.
All reported classes pass KS-tests with p-values far below 0.05 (sampled at 1M trades). Spread and volatility classes are regime-dependent: tight-spread / low-vol buckets dominate the calm train period, wide-spread / high-vol buckets dominate the stressed validation period (Feb 2026 crash). Literature features (VPIN, Amihud) are more stable across both periods.

## Practical Implications for Filter Construction.
1. Core features for the ML filter: distance to mid, VPIN, liquidation flow, trade flow.
2. Use a tree-based model - distance to mid, spread, and volatility have non-linear (U/V-shaped) relationships with markout.
3. Drop microprice deviation (redundant with OBI on BBO data), keep OBI.
4. Liquidation flow from both exchanges adds orthogonal information to price-based signals.
5. The 500k USD/day turnover constraint is easily satisfied - the strongest classes each cover 11–33% of turnover.

## Limitations.
- **Cross-symbol stability (ETH) not tested** due to time constraints. BTC and ETH perpetuals share microstructure and correlated liquidation dynamics (Task 1), so classes are expected to generalize, but this is unverified.
- **LOB-only data**: no order book depth, so microprice deviation collapses to OBI and deeper-level imbalance features cannot be tested.
- **Quantile boundaries computed on the full sample**: production use would require online (EMA-based) estimation to avoid look-ahead in thresholds.
- **Three-month-per-period split**: includes one major crash (Feb 2026) but may not represent all regimes.

## References.
- Cont, Kukanov, Stoikov (2014). The Price Impact of Order Book Events.
- Easley, López de Prado, O'Hara (2012). Flow Toxicity and Liquidity in a High-Frequency World.
- Stoikov (2018). The Micro-Price: A High-Frequency Estimator of Future Prices.
- Amihud (2002). Illiquidity and Stock Returns.
- Lopez de Prado (2018). Advances in Financial Machine Learning.